## Week 6, Day 3 exercise: add another context source

I add the official Time MCP server (`mcp-server-time`) as a new context source. A model has no clock, so this gives the agent the current time in any timezone and lets it convert between timezones.

**Pattern used: 2, web search (fresh information plus a tool filter).** Like Tavily, it brings in information the model cannot know from training. The steps are the same: connect a ready-made server, list its tools, then give the agent only the tools it needs.

In [ ]:
# Step 0: imports and API keys
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
from IPython.display import Markdown, display

load_dotenv(override=True)

In [ ]:
# Step 1: tell the SDK how to launch the server.
# uvx downloads and runs the Python package mcp-server-time in its own isolated environment.
# No API key is needed. The server talks to us over stdin/stdout (stdio).
time_params = {"command": "uvx", "args": ["mcp-server-time"]}

# Start the server once, only to ask which tools it offers.
async with MCPServerStdio(params=time_params, client_session_timeout_seconds=60) as server:
    time_tools = await server.list_tools()

[tool.name for tool in time_tools]

In [ ]:
# Step 2: describe the job.
# The server offers three tools; we keep the two we need. Giving an agent only the tools
# it needs is itself context engineering.
only_time = create_static_tool_filter(allowed_tool_names=["get_current_time", "convert_time"])

instructions = "You answer questions about time and timezones. Always use your tools, never guess the time."
request = "What time is it now in Lima, Peru and in Madrid, Spain? And if a meeting is at 15:00 in Lima today, what time is that in Madrid?"
model = "gpt-5.4-mini"

In [ ]:
# Step 3: run the agent with the Time server attached.
# The agent reads the tool list, decides to call get_current_time and convert_time,
# and answers from their results. The trace appears at platform.openai.com/traces.
async with MCPServerStdio(params=time_params, client_session_timeout_seconds=60, tool_filter=only_time) as mcp_server:
    agent = Agent(name="time_assistant", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("time_assistant"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))